In [ ]:
# 셀 1: 파일 읽기 + 확인
import pandas as pd

df = pd.read_csv('성분표.csv', encoding='euc-kr')
print(df.shape)
print(df.columns.tolist())
df.head(3)

In [ ]:
# 셀 2: OpenAI로 target_skin 추출
import openai
import json
import time

client = openai.OpenAI(api_key='YOUR_API_KEY')

def get_target_skin(product_name, ingredients):
    prompt = f"""
당신은 화장품 성분 전문가입니다.
아래 화장품의 전성분을 분석해서 어떤 피부 타입에 적합한지 판단해주세요.

제품명: {product_name}
전성분: {ingredients}

다음 JSON 형식으로만 답해주세요. 다른 말은 하지 마세요:
{{
  "target_skin": ["건성", "지성", "복합성", "민감성", "여드름성"] 중 해당하는 것만 배열로,
  "reason": "30자 이내로 핵심 이유"
}}
"""
    try:
        res = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        text = res.choices[0].message.content.strip()
        # ```json 제거
        text = text.replace('```json', '').replace('```', '').strip()
        return json.loads(text)
    except Exception as e:
        return {"target_skin": [], "reason": f"오류:{str(e)}"}

print("함수 준비 완료!")

In [ ]:
# 셀 3: 전체 실행 (제품별 중복 제거 후 처리)
results = []

for i, row in df.iterrows():
    result = get_target_skin(row['제품명'], row['전성분'])
    results.append({
        '제품명': row['제품명'],
        '브랜드': row.get('브랜드', ''),
        '카테고리': row.get('카테고리', ''),
        'target_skin': ', '.join(result.get('target_skin', [])),
        'reason': result.get('reason', '')
    })

    if (i + 1) % 10 == 0:
        print(f"[{i+1}/{len(df)}] {row['제품명'][:30]} → {result.get('target_skin')}")

    # 50개마다 중간 저장
    if (i + 1) % 50 == 0:
        pd.DataFrame(results).to_csv('target_skin_result.csv', index=False, encoding='utf-8-sig')
        print(f"  💾 중간 저장 완료")

    time.sleep(0.5)  # API 요청 제한 방지

# 최종 저장
result_df = pd.DataFrame(results)
result_df.to_csv('target_skin_result.csv', index=False, encoding='utf-8-sig')
print(f"\n✅ 완료! target_skin_result.csv 저장됨")
result_df.head()